# Hybrid Retrieval with a Reranker

## Load in documents and chunk them

In [1]:
from rag.documents import load_documents
from rag.chunk import chunk_documents

FILE_PATH = "/home/nick/github-projects/Sec-Rag/data/google_10K.pdf"

documents = load_documents(FILE_PATH)

chunks = chunk_documents(
    documents=documents,
    chunk_size=400,
    chunk_overlap=40
)

print(f"Length of Documents: {len(documents)}")
print(f"Length of Chunks: {len(chunks)}")

/home/nick/github-projects/Sec-Rag/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of Documents: 107
Length of Chunks: 1037


## Build the hybrid retriever

In [2]:
from rag.bm25 import BM25Retriever
from rag.dense import DenseRetriever
from rag.hybrid import HybridRetriever

bm25 = BM25Retriever()
bm25.add_documents(chunks)

dense = DenseRetriever()
dense.add_documents(chunks)

hybrid = HybridRetriever(bm25=bm25, dense=dense)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11279.43it/s]


## Wrap it in the reranker

In [3]:
from rag.reranker import RerankerRetriever

reranked = RerankerRetriever(base_retriever=hybrid, candidate_pool=30)

## Hybrid before and after reranking

In [4]:
def pages(docs):
    return [d.metadata["page"] for d in docs]


query = "how much did Google spend on research and development in 2025"

hybrid_docs = hybrid.retrieve(query, top_k=4)
reranked_docs = reranked.retrieve(query, top_k=4)

print(f"hybrid pages:   {pages(hybrid_docs)}")
print(f"reranked pages: {pages(reranked_docs)}")

for doc in reranked_docs:
    print(f"\n[page {doc.metadata['page']}]\n{doc.page_content[:200]}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5256.90it/s]


hybrid pages:   [41, 42, 39, 32]
reranked pages: [42, 41, 39, 40]

[page 42]
Other Bets (4,444) (7,515)
Alphabet-level activities (10,541) (16,760)
Total income from operations $ 112,390  $ 129,039 
Alphabet-level activities primarily reflect expenses related to our shared AI 

[page 41]
costs, largely for YouTube, depreciation expense, and other technical infrastructure operations costs.
Research and Development
The following table presents research and development expenses (in milli

[page 39]
Total revenues $ 350,018  $ 402,836 
Google Services
Google Advertising
Google Search & other
Google Search & other revenues increased $26.4 billion from 2024 to 2025. The overall growth was driven by

[page 40]
changes in device mix, geographic mix, advertiser spending, ongoing product and policy changes, product mix,
property mix, and changes in foreign currency exchange rates.
Google Subscriptions, Platfor


## Answer over the reranked context

In [5]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from rag.llm import DEFAULT_CHAT_MODEL, LangChainGenerator
from rag.pipeline import RAGPipeline

load_dotenv()

generator = LangChainGenerator(ChatOpenAI(model=DEFAULT_CHAT_MODEL, max_tokens=512))
pipeline = RAGPipeline(reranked, generator, top_k=4)

for i, q in enumerate([
    "how much did Google spend on research and development in 2025?",
    "how much did Google spend on research and development in 2024?",
    "what were Google's main operating costs?",
    "what are some pending acquisitions the company faces?",
]):
    answer = pipeline.answer(q)
    print(f"\n{i + 1} Q: {q}")
    print(f"hybrid pages:   {pages(hybrid.retrieve(q, top_k=4))}")
    print(f"reranked pages: {pages(answer.documents)}")
    print(f"> {answer.text}")


1 Q: how much did Google spend on research and development in 2025?
hybrid pages:   [41, 42, 39, 32]
reranked pages: [42, 41, 39, 40]
> Google spent $61,087 million on research and development in 2025. This information is sourced from the context on page 41 of the SEC 10-K filings.

2 Q: how much did Google spend on research and development in 2024?
hybrid pages:   [41, 39, 42, 40]
reranked pages: [42, 41, 39, 66]
> Google spent $49,326 million on research and development in 2024. [Source: /home/nick/github-projects/Sec-Rag/data/google_10K.pdf p41]

3 Q: what were Google's main operating costs?
hybrid pages:   [42, 38, 94, 95]
reranked pages: [94, 42, 36, 95]
> Google's main operating costs include research and development, sales and marketing, and general and administrative expenses. Additionally, costs related to technical infrastructure, office facilities usage, content acquisition, traffic acquisition costs (TAC), and device costs are also significant components of their operating

In [6]:
questions = [
    "how much did Google spend on research and development in 2025?",
    "how much did Google spend on research and development in 2024?",
    "what were Google's main operating costs?",
    "what are some pending acquisitions the company faces?",
]


def rerank_scores(query, docs):
    pairs = [(query, d.page_content) for d in docs]
    return reranked.cross_encoder.predict(pairs)


for q in questions:
    docs = reranked.retrieve(q, top_k=4)
    print(f"\nQ: {q}")
    for doc, score in zip(docs, rerank_scores(q, docs)):
        print(f"page {doc.metadata['page']}: {score:.3f}")


Q: how much did Google spend on research and development in 2025?
page 42: 6.099
page 41: 4.389
page 39: 3.563
page 40: 2.879

Q: how much did Google spend on research and development in 2024?
page 42: 6.068
page 41: 4.358
page 39: 3.709
page 66: 1.658

Q: what were Google's main operating costs?
page 94: 3.225
page 42: 2.953
page 36: 1.429
page 95: 0.770

Q: what are some pending acquisitions the company faces?
page 81: 1.382
page 46: 0.687
page 27: -2.093
page 39: -4.878


In [7]:
from rag.metrics.evaluate import EvalExample, evaluate_retriever

relevant_pages = [{41}, {41}, {36, 40}, {46, 81}]
examples = [EvalExample(q, pages) for q, pages in zip(questions, relevant_pages)]

for name, retriever in [("hybrid", hybrid), ("reranked", reranked)]:
    result = evaluate_retriever(retriever, examples, k=4)
    print(f"\n{name}")
    for metric, score in result.aggregate.items():
        print(f"  {metric}: {score:.3f}")


hybrid
  hit@4: 0.750
  precision@4: 0.250
  recall@4: 0.750
  mrr: 0.583
  ndcg@4: 0.643
  map: 0.604

reranked
  hit@4: 1.000
  precision@4: 0.312
  recall@4: 0.875
  mrr: 0.583
  ndcg@4: 0.642
  map: 0.542
